- references
    - https://www.youtube.com/watch?v=jYn_1PpRzxI

### skip/residual connection

> the representation power of a deep neural network comes from stacking non-linear layers.

- 直觉上，更深的模型应该能学到更 complex & expressive representations 并产生更好的结果；
    - 但在 resnet 之前，更深的网络，更难训练，不见得比 shallow nn 更好的 training loss or test loss

$$
x_{\ell+1}=x_\ell+\mathcal F_{w_\ell}(x_\ell)
$$

- $x_2=x_1+\mathcal F_{w_1}(x_1)$
    - $\mathcal F_{w_1}(x_1)=x_2-x_1$，刻画着 residual function，$\mathcal F$ 可以是 cnn/attention/ffn
- $x_3=x_2+\mathcal F_{w_2}(x_2)=(x_1+\mathcal F_{w_1}(x_1))+\mathcal F_{w_2}(x_2)$
    - $\mathcal F_{w_2}(x_2)=x_3-x_2$
- $x_4=x_3+\mathcal F_{w_3}(x_3)=((x_1+\mathcal F_{w_1}(x_1))+\mathcal F_{w_2}(x_2))+\mathcal F_{w_3}(x_3)$
    - $\mathcal F_{w_3}(x_3)=x_4-x_3$

$$
x_L=x_\ell+\sum_{i=\ell}^{L-1}\mathcal F_{w_i}(x_i)
$$

- identity mapping：features can be directly propagated from any shallow (earlier) layer ($\ell$) to any deeper (later) layer ($L$)
    - Identity Mappings in Deep Residual Networks
- 梯度及性质: $\mathcal E$

$$
\begin{split}
\frac{\partial \mathcal E}{\partial x_\ell} &= \frac{\partial \mathcal E}{\partial x_L} \cdot \frac{\partial x_L}{\partial x_\ell}\\
&=\frac{\partial \mathcal E}{\partial x_L}\cdot\left\{I + \frac{\partial}{\partial x_\ell} \sum_{i=\ell}^{L-1}\mathcal F_{w_i}(x_i)\right\}\\
&=\underbrace{\frac{\partial \mathcal E}{\partial x_L}}_{\text{项1: 直接梯度}} + \underbrace{\frac{\partial \mathcal E}{\partial x_L} \left( \frac{\partial}{\partial x_\ell} \sum_{i=\ell}^{L-1}\mathcal F_{w_i}(x_i) \right)}_{\text{项2: 经过权重的梯度}}
\end{split}
$$

- 项1 $\frac{\partial \mathcal E}{\partial x_L}$。这表明，深层（$L$）的梯度信息，可以不经过任何权重矩阵的衰减，直接“跳跃”回传到浅层（$\ell$）。
    - 在普通的神经网络（Plain Net）中，梯度必须经过每一层的乘法（$\prod W_i$）。
        - $\frac{\partial x_L}{\partial x_\ell} = \prod_{i=\ell}^{L-1} W_i$
        - 如果 $W_i$ 很小（例如小于1），连乘会导致梯度呈指数级衰减（Vanishing Gradient）。
    - 在 ResNet 中，由于恒等映射（Identity Mapping）的存在，梯度可以通过 $1$（即 $I$）这条通道无损传输。
        - 梯度是加法形式：$\frac{\partial x_L}{\partial x_\ell} = I + \text{residual\_gradients}$

### why skip conn

在没有残差链接（也就是传统的平原网络 Plain Network）时，假设网络中的某几个堆叠层（Stacked layers）试图学习一个理想的底层映射（Underlying mapping），我们将其记为 $\mathcal{H}(x)$。此时，网络被迫让这些层的权重直接去拟合这个完整的 $\mathcal{H}(x)$，这就是所谓的原来的 projection（原始投影/映射）。而在引入残差链接（Skip connection / Shortcut）后，网络结构发生了一个极其巧妙的转换：网络不再直接强求这几个层去拟合 $\mathcal{H}(x)$，而是让它们去拟合一个残差映射（Residual mapping），记为 $\mathcal{F}(x)$。它们之间的数学关系被定义为：

$$\mathcal{F}(x) = \mathcal{H}(x) - x$$

通过残差链接，前向传播的实际输出变成了：

$$y = \mathcal{F}(x) + x$$

由于 $x$ 是通过旁路（Shortcut）直接恒等映射（Identity mapping）过来的，那么原本的那几个堆叠层，现在实际上只需要学习 $\mathcal{F}(x)$ 即可。

#### skip/res conn in transformer

> $\mathcal F_{w_\ell}(x_\ell)=x_{\ell+1}-x_\ell$，刻画着 residual function，$\mathcal F$ 可以是 cnn/attention/ffn
- https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/11_qwen3/standalone-qwen3.ipynb
    - shortcut connection

```python
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    def forward(self, x, mask, cos, sin):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # Add the original input back

        return x
```


#### $W$ 初始化很小，会导致 $\mathcal{F}$ 的导数接近 0？

- $\mathcal{F}(x) = W \cdot x$,
    - $\frac{\partial \mathcal{F}}{\partial x} = W$
    - 向量对向量 grad => 雅可比矩阵
- $\mathcal{F}(x) = \sigma(W \cdot x)$
    - $\frac{\partial \mathcal{F}}{\partial x} = \underbrace{\text{diag}(\sigma'(W \cdot x))}_{\text{激活函数的导数}} \cdot \underbrace{W}_{\text{权重矩阵}}$
    - 权重矩阵 $W$ 是以“乘法因子”的身份出现在导数表达式里的。
- 权重 $W$ 决定了输入 $x$ 的变化能多大程度地影响输出。如果 $W$ 里的数值全是 0.001（非常小，这里不严谨，应该要求的是谱范数），那么 $x$ 变动 1，输出 $W \cdot x$ 只变动 0.001。这种微弱的变化意味着“导数”非常小。
- 公式解释：在上面的公式中，如果我们将 $W$ 初始化为非常接近 0 的随机分布（例如高斯分布 $N(0, 0.0001)$），那么梯度表达式里那一项 $W$ 就会把整个梯度的数值拉得极低。

$$
\frac{\partial \mathcal E}{\partial x_\ell} = \frac{\partial \mathcal E}{\partial x_L} \left( I + \frac{\partial \mathcal F}{\partial x_\ell} \right)
$$
如果我们将权重 $W$ 初始化得很小（例如 $10^{-5}$）：对于普通网络（没有 $I$）：梯度是连乘 $\prod W_i$。很多个 $10^{-5}$ 乘在一起，梯度直接归零，网络学不动。
对于 ResNet：因为 $\frac{\partial \mathcal F}{\partial x_\ell}$ 里包含 $W$（作为乘数），所以 $\frac{\partial \mathcal F}{\partial x_\ell} \approx 0$。此时梯度变成：
$$
\frac{\partial \mathcal E}{\partial x_\ell} \approx \frac{\partial \mathcal E}{\partial x_L} \cdot (I + 0) = \frac{\partial \mathcal E}{\partial x_L}
$$

即使权重 $W$ 小到让 $\mathcal{F}$ 分支完全“断路”，ResNet 依然有一条 $I$（恒等映射）保底，让梯度能原样传回来。这就是为什么 ResNet 即使初始化很小也能训练，而普通网络不行。

### single path => parallel paths

In [1]:
from IPython.display import Image

In [5]:
Image(url='./imgs/hc.png', width=600)

$$
\mathbf{x}_l = \begin{bmatrix}
\mathbf{x}_l^{(1)} \\
\mathbf{x}_l^{(2)} \\
\mathbf{x}_l^{(3)} \\
\mathbf{x}_l^{(4)}
\end{bmatrix} \in \mathbb{R}^{n \times d}
$$
- $\mathbf{x}_l\in \mathbb{R}^{n \times d}$ input to the $\ell$-th layer  
- $\mathbf{x}_l^{pre}=\mathcal H_{\ell}^{pre}\mathbf x_\ell\in \mathbb R^{1\times d}$
    - $\mathcal H_{\ell}^{pre}=[\alpha_\ell^{(1)},\alpha_\ell^{(2)},\alpha_\ell^{(3)},\alpha_\ell^{(4)}]\in \mathbb R^{1\times n}$
    - aggregate streams
- $\mathbf z_\ell=\mathcal H_\ell^{post}\mathcal F_{w_\ell}(\mathbf x_\ell^{pre})\in \mathbb R^{n\times d}$
    - $\mathcal H_{\ell}^{post}=[\beta_\ell^{(1)},\beta_\ell^{(2)},\beta_\ell^{(3)},\beta_\ell^{(4)}]^T\in\mathbb R^{n\times 1}$
    - expand streams
- $h_\ell=\mathcal H_\ell^{res}\mathbf x_\ell\in \mathbb R^{n\times d}$
    - $\mathcal H_\ell^{res}\in \mathbb R^{n\times n}$
    - mix streams
- $\mathbf x_{\ell+1}=h_\ell+z_\ell$
    - $z_\ell$ as the residual

### 双随机矩阵

- 谱范数恒为1。（矩阵连乘，不会消失也不会爆炸）
- 乘法封闭性：两个双随机矩阵乘得到的还是双随机矩阵。
- 从几何角度看，双随机矩阵构成的集合称为 Birkhoff 多面体 (Birkhoff Polytope)。

In [6]:
import torch

def generate_doubly_stochastic(n, max_iter=1000, tol=1e-6):
    """
    使用 Sinkhorn-Knopp 算法生成一个 n x n 的双随机矩阵
    """
    # 初始化一个非负矩阵
    A = torch.rand(n, n)
    
    for _ in range(max_iter):
        # 行归一化
        r_sum = A.sum(dim=1, keepdim=True)
        if torch.allclose(r_sum, torch.ones_like(r_sum), atol=tol) and \
           torch.allclose(A.sum(dim=0), torch.ones(n), atol=tol):
            break
        A = A / r_sum
        
        # 列归一化
        c_sum = A.sum(dim=0, keepdim=True)
        A = A / c_sum
        
    return A

/home/zhangchunhui/miniconda3/envs/verl/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [7]:
N = 10
torch.manual_seed(42) # 固定随机种子以便复现

print(f"--- 验证 {N}x{N} 矩阵乘法封闭性 ---")

# 1. 生成两个双随机矩阵 A 和 B
A = generate_doubly_stochastic(N)
B = generate_doubly_stochastic(N)

# 2. 计算乘积 C
C = torch.matmul(A, B)

--- 验证 10x10 矩阵乘法封闭性 ---


In [8]:
# 3. 验证 C 是否为双随机矩阵
# 条件1: 元素非负 (由于 A, B 非负，C 必然非负，检查是否大于 -epsilon 即可)
is_non_negative = torch.all(C >= -1e-6) 

# 条件2: 行和为 1
row_sums = C.sum(dim=1)
is_row_stochastic = torch.allclose(row_sums, torch.ones(N), atol=1e-5)

# 条件3: 列和为 1
col_sums = C.sum(dim=0)
is_col_stochastic = torch.allclose(col_sums, torch.ones(N), atol=1e-5)

In [9]:
is_non_negative, is_row_stochastic, is_col_stochastic

(tensor(True), True, True)